# DREIDING: verifying the `xnn` implementation against LAMMPS and the 1990 paper

The `xnn` DREIDING model is a clean-room implementation of the published
functional form (Mayo, Olafson & Goddard III, *J. Phys. Chem.* **94**,
8897, 1990). This notebook verifies it three independent ways:

1. **the published parameter tables** — the per-atom generators read from
   the SEAMM `dreiding.frc` are checked against Tables I, II and III of the
   paper (bond radii, equilibrium angles, van der Waals $R_0$ / $D_0$ /
   $\zeta$, and the universal force constants);
2. **numerical parity with LAMMPS** — every valence term the `xnn` rule
   engine generates is written into a LAMMPS data file and evaluated by
   LAMMPS's own DREIDING styles (`bond_style harmonic`, `angle_style
   cosine/squared` and `cosine`, `dihedral_style harmonic`, `improper_style
   umbrella`, `pair_style lj/cut` / `buck` / `hbond/dreiding/lj`), which are
   an entirely independent implementation of the same expressions. Energies
   are compared **term by term** and forces atom by atom, on randomized
   conformations of nine molecules covering every rule branch, for both the
   Lennard-Jones and exponential-6 nonbond forms;
3. **the paper's own numbers** — the eclipsed-ethane barrier is exactly the
   published total $V_{JK} = 2.0$ kcal/mol of eq 14, and a linear
   donor-hydrogen-acceptor bridge at $R = R_{hb}$ sits exactly at $-D_{hb}$,
   the analytic minimum of eq 38. (Relaxed rotational barriers and
   conformational energies, Tables XI and XII, are reproduced in
   `examples/ffnn/dreiding/dreiding_conformational_energetics.ipynb`.)

What makes DREIDING different from a tabulated force field such as OPLS is
that *nothing* bonded is tabulated: bond lengths come from additive atomic
radii (eq 6), the stretch and bend force constants are single universal
numbers (eqs 7 and 12), and every torsion barrier follows from the
hybridizations of the two central atoms plus the bond order between them
(eqs 14-23). So the comparison below tests the **rule engine** as much as
the energy expressions -- LAMMPS is given the coefficients `xnn` generated,
and any error in a rule would show up as a wrong barrier rather than a
mismatch. The paper's own barrier tables (check 3 and the companion
notebook) close that loop.

Unit conventions are part of what is verified: `xnn` stores DREIDING
parameters in the paper's units (kcal/mol, Angstrom, degrees) and converts
to eV / radians when it assembles tensors, so every comparison below
converts back with the same constant the model uses.

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
import math
import os
import tempfile

import numpy as np
import torch

torch.set_default_dtype(torch.float64)
rng = np.random.default_rng(7)

import lammps

import xnn
from xnn.common.data import structure_to_graph
from xnn.common.models import ForceStressOutput
from xnn.ffnn.models import Dreiding, MolecularTopology, read_dreiding
from xnn.ffnn.models.dreiding import KE, RULE_IDS
from xnn.ffnn.models.dreidinglib import TORSION_RULES
from xnn.ffnn.models.oplslib import KCAL_TO_EV

EV_TO_KCAL = 1.0 / KCAL_TO_EV
DEG = math.pi / 180.0
print("xnn:", xnn.__version__, "| torch:", torch.__version__,
      "| LAMMPS:", lammps.lammps(cmdargs=["-log", "none", "-screen", "none"]).version())

xnn: 0.2.0 | torch: 2.5.1+cu121 | LAMMPS: 20250722


## 1. The parameter tables of the paper

The SEAMM `dreiding.frc` shipped with `xnn` carries the per-atom
generators; two `#define` variants select the nonbond form
(`dreiding` = Lennard-Jones eq 31', `dreiding/X6` = exponential-6 eq 32').
Everything below is checked against the printed tables.

In [2]:
lj = read_dreiding("dreiding")
x6 = read_dreiding("dreiding/X6")

# Table I: bond radius R0 (A) and the equilibrium angle theta0 (deg) of the
# atom as an angle *center*
TABLE_I = {"H_": (0.330, 180.0), "H__b": (0.510, 90.0), "B_3": (0.880, 109.471),
           "C_3": (0.770, 109.471), "C_R": (0.700, 120.0), "C_2": (0.670, 120.0),
           "C_1": (0.602, 180.0), "N_3": (0.702, 106.7), "N_R": (0.650, 120.0),
           "O_3": (0.660, 104.51), "O_2": (0.560, 120.0), "F_": (0.611, 180.0),
           "Si3": (0.937, 109.471), "P_3": (0.890, 93.3), "S_3": (1.040, 92.1),
           "Cl": (0.997, 180.0), "Ga3": (1.210, 109.471), "Br": (1.167, 180.0),
           "Sn3": (1.373, 109.471), "I_": (1.360, 180.0), "Na": (1.860, 90.0),
           "Ca": (1.940, 90.0), "Fe": (1.285, 90.0), "Zn": (1.330, 109.471)}
bad = [(t, lj.radius[t], lj.theta0[t]) for t, (r, a) in TABLE_I.items()
       if abs(lj.radius[t] - r) > 1e-9 or abs(lj.theta0[t] - a) > 1e-9]
print(f"Table I  : {len(TABLE_I)} atom types checked, {len(bad)} mismatches", bad or "")

# Table II: van der Waals R0 (A), D0 (kcal/mol) and the X6 scaling zeta
TABLE_II = {"H_": (3.195, 0.0152, 12.382), "B_3": (4.02, 0.095, 14.23),
            "C_3": (3.8983, 0.0951, 14.034), "N_3": (3.6621, 0.0774, 13.843),
            "O_3": (3.4046, 0.0957, 13.483), "F_": (3.4720, 0.0725, 14.444),
            "Cl": (3.9503, 0.2833, 13.861), "S_3": (4.0300, 0.3440, 12.0),
            "Br": (3.95, 0.37, 12.0), "I_": (4.15, 0.51, 12.0),
            "Na": (3.144, 0.5, 12.0), "Zn": (4.54, 0.055, 12.0),
            "C_R1": (4.23, 0.1356, 14.034), "C_34": (4.2370, 0.3016, 12.0),
            "C_33": (4.1524, 0.25, 12.0), "C_32": (4.0677, 0.1984, 12.0),
            "C_31": (3.9830, 0.1467, 12.0)}
bad = [t for t, (r, d, z) in TABLE_II.items()
       if abs(lj.vdw_r0[t] - r) > 1e-9 or abs(lj.vdw_d0[t] - d) > 1e-9
       or abs(x6.x6_zeta[t] - z) > 1e-9]
print(f"Table II : {len(TABLE_II)} atom types checked, {len(bad)} mismatches", bad or "")
# the two variants share R0 and D0; only the functional form differs
assert lj.vdw_r0 == x6.vdw_r0 and lj.vdw_d0 == x6.vdw_d0

# Table III: the universal valence force constants and the inversions
print(f"Table III: K_bond(1) = {lj.bond_k1} kcal/mol/A^2, D_bond(1) = {lj.bond_d1} kcal/mol,"
      f" K_angle = {lj.angle_k} kcal/mol/rad^2, delta = {lj.delta} A")
print(f"           inversions: {sorted(lj.oop)}")
print(f"           planar centers K = {lj.oop['C_2'][0]} kcal/mol/rad^2 at Psi0 = {lj.oop['C_2'][1]} deg;"
      f" C_31 at Psi0 = {lj.oop['C_31'][1]} deg")
# Table V: the hydrogen bond, in the paper's no-charges convention
print(f"Table V  : D_hb = {lj.hbond_d0} kcal/mol, R_hb = {lj.hbond_r0} A")
# eqs 14-23: the nine torsion rules, as (total barrier V, periodicity n, phi0)
print("\neqs 14-23, the torsion rules (V kcal/mol, n, phi0 deg):")
for r in RULE_IDS:
    print(f"   ({r})  {TORSION_RULES[r]}")

Table I  : 24 atom types checked, 0 mismatches 
Table II : 17 atom types checked, 0 mismatches 
Table III: K_bond(1) = 700.0 kcal/mol/A^2, D_bond(1) = 70.0 kcal/mol, K_angle = 100.0 kcal/mol/rad^2, delta = 0.01 A
           inversions: ['B_2', 'C_2', 'C_31', 'C_R', 'N_2', 'N_R', 'O_2', 'O_R']
           planar centers K = 40.0 kcal/mol/rad^2 at Psi0 = 0.0 deg; C_31 at Psi0 = 54.74 deg
Table V  : D_hb = 9.0 kcal/mol, R_hb = 2.75 A

eqs 14-23, the torsion rules (V kcal/mol, n, phi0 deg):
   (a)  (2.0, 3, 180.0)
   (b)  (1.0, 6, 0.0)
   (c)  (45.0, 2, 180.0)
   (d)  (25.0, 2, 180.0)
   (e)  (5.0, 2, 180.0)
   (f)  (10.0, 2, 180.0)
   (h)  (2.0, 2, 90.0)
   (i)  (2.0, 2, 180.0)
   (j)  (2.0, 3, 180.0)


## 2. A LAMMPS twin of the generated force field

LAMMPS implements the DREIDING expressions natively, so the translation is
one-to-one up to how each code factors the constants:

| DREIDING (paper) | `xnn` | LAMMPS style | coefficient |
|---|---|---|---|
| eq 4a, $\tfrac12 k_n (R-R_0)^2$ | `e_bond` | `bond_style harmonic`, $K(R-R_0)^2$ | $K = \tfrac12 n \cdot 700$ |
| eq 10a, $\tfrac12 \frac{K}{\sin^2\theta_0}(\cos\theta - \cos\theta_0)^2$ | `e_angle` | `angle_style cosine/squared`, $K(\cos\theta-\cos\theta_0)^2$ | $K = \tfrac12 \cdot 100/\sin^2\theta_0$ |
| eq 10', $K(1+\cos\theta)$ (linear centers) | `e_angle` | `angle_style cosine`, $K(1+\cos\theta)$ | $K = 100$ |
| eq 13, $\tfrac12 V\{1-\cos[n(\phi-\phi_0)]\}$ | `e_torsion` | `dihedral_style harmonic`, $K[1+d\cos(n\phi)]$ | $K=\tfrac12 V$, $d=-\cos(n\phi_0)$ |
| eq 28a/28c | `e_inversion` | `improper_style umbrella` | $K_{\text{LAMMPS}} = K/3$ |
| eq 31', $D_0[\rho^{-12}-2\rho^{-6}]$ | `e_vdw` | `pair_style lj/cut` | $\sigma = R_0/2^{1/6}$, $\epsilon = D_0$ |
| eq 32', exponential-6 | `e_vdw` | `pair_style buck`, $A e^{-r/\rho} - C/r^6$ | $A,\ \rho = 1/C_{ij},\ C$ from $D_0, R_0, \zeta$ |
| eq 38, 12-10 $\cos^4\theta$ | `e_hbond` | `pair_style hbond/dreiding/lj` with $n=4$ | $\epsilon = D_{hb}$, $\sigma = R_{hb}$ |

Two conventions deserve a note.

* **The inversion factor of 3.** The paper adds "all three possible
  inversion terms with each weighted by a factor of $1/3$" at every
  three-coordinate center, because eq 28 treats the $IL$ bond differently
  from $IJ$ and $IK$. `xnn` generates all three axis choices and divides by
  three; LAMMPS applies each listed improper once, so its $K$ is $K/3$.
* **The Coulomb constant.** Eq 37 specifies 332.0637 kcal·Å/mol, while
  LAMMPS `real` units use `qqr2e = 332.06371` (a relative difference of
  $3\times10^{-8}$). Scaling the charges written into the data file by
  $\sqrt{332.0637/332.06371}$ makes the two Coulomb terms identical, so the
  comparison isolates the functional forms rather than re-measuring a
  constant. `xnn` keeps the published value.

Also note `special_bonds lj/coul 0.0 0.0 1.0`: DREIDING excludes 1,2 and
1,3 nonbonded pairs but counts **1,4 pairs in full**, unlike OPLS.

In [3]:
def _r(x, nd=15):
    return round(float(x), nd)


def lammps_twin(model, positions, box=60.0):
    '''Return (data-file text, command list) evaluating the same DREIDING model.'''
    ff, top = model.ff, model.topology
    P = {k: v.detach() for k, v in ff.params.items()}
    n, cutoff = top.n_atoms, float(model.cutoff)
    used = list(dict.fromkeys(top.types))
    at_id = {t: i + 1 for i, t in enumerate(used)}
    ti = {t: ff.type_index(t) for t in used}
    orders = list(top.bond_orders) or [1.0] * len(top.bonds)

    # --- bonds: eq 6 for R0, eq 9a for the bond-order scaling -------------
    b_rows, b_types = [], {}
    for (i, j), o in zip(top.bonds, orders):
        r0 = float(P["radius"][ti[top.types[i]]]
                   + P["radius"][ti[top.types[j]]]) - ff.delta
        key = (_r(0.5 * o * float(P["bond_k"]) * EV_TO_KCAL), _r(r0))
        b_types.setdefault(key, len(b_types) + 1)
        b_rows.append((b_types[key], i + 1, j + 1))

    # --- angles: eq 10a, or eq 10' at a linear center ---------------------
    a_rows, a_types = [], {}
    ka = float(P["angle_k"]) * EV_TO_KCAL
    for m, (i, j, k) in enumerate(top.angles):
        th0 = float(P["theta0"][ti[top.types[j]]])
        key = ("cosine", _r(ka), 0.0) if bool(model.angle_linear[m]) else \
              ("cosine/squared", _r(0.5 * ka / math.sin(th0) ** 2), _r(th0 / DEG))
        a_types.setdefault(key, len(a_types) + 1)
        a_rows.append((a_types[key], i + 1, j + 1, k + 1))

    # --- torsions: eq 13, barrier already split over the central bond -----
    d_rows, d_types = [], {}
    for m in range(model.dihedral_index.shape[1]):
        i, j, k, l = [int(x) for x in model.dihedral_index[:, m]]
        rule = int(model.dihedral_rule[m])
        v = float(model.dihedral_weight[m]) * float(P["torsion_v"][rule]) * EV_TO_KCAL
        key = (_r(0.5 * v), -int(ff.rule_sign[rule]), int(ff.rule_n[rule]))
        d_types.setdefault(key, len(d_types) + 1)
        d_rows.append((d_types[key], i + 1, j + 1, k + 1, l + 1))

    # --- inversions: eq 28, each of the three axes carries K/3 ------------
    i_rows, i_types = [], {}
    for m in range(model.inversion_index.shape[1]):
        c, a, b, d = [int(x) for x in model.inversion_index[:, m]]
        idx = ti[top.types[c]]
        key = (_r(float(P["oop_k"][idx]) * EV_TO_KCAL / 3.0),
               _r(float(P["oop_psi0"][idx]) / DEG))
        i_types.setdefault(key, len(i_types) + 1)
        i_rows.append((i_types[key], c + 1, a + 1, b + 1, d + 1))

    # --- data file --------------------------------------------------------
    pos = np.asarray(positions, dtype=float)
    q = model.charge.detach().cpu().numpy() * math.sqrt(332.0637 / 332.06371)
    L = ["DREIDING model generated by xnn", "",
         f"{n} atoms", f"{len(b_rows)} bonds", f"{len(a_rows)} angles",
         f"{len(d_rows)} dihedrals", f"{len(i_rows)} impropers", "",
         f"{len(used)} atom types", f"{len(b_types)} bond types",
         f"{len(a_types)} angle types", f"{len(d_types)} dihedral types",
         f"{len(i_types)} improper types", "",
         f"{-box} {box} xlo xhi", f"{-box} {box} ylo yhi",
         f"{-box} {box} zlo zhi", "", "Masses", ""]
    L += [f"{tid} {float(ff.type_mass[ti[t]]) or 1.0}   # {t}"
          for t, tid in at_id.items()]
    L += ["", "Atoms # full", ""]
    L += [f"{a+1} 1 {at_id[top.types[a]]} {q[a]:.14f} "
          f"{pos[a,0]:.14f} {pos[a,1]:.14f} {pos[a,2]:.14f}" for a in range(n)]
    for name, rows in (("Bonds", b_rows), ("Angles", a_rows),
                       ("Dihedrals", d_rows), ("Impropers", i_rows)):
        if rows:
            L += ["", name, ""] + [" ".join(str(x) for x in (m + 1, *row))
                                   for m, row in enumerate(rows)]

    # --- commands ---------------------------------------------------------
    hb = model.use_hbond and model.hbond_index.shape[1] > 0
    kernel = "buck" if ff.form == "x6" else "lj/cut"
    base = f"{kernel}{'/coul/cut' if model.has_coulomb else ''} {cutoff}"
    hb_ang = math.acos(model.hbond_cos) / DEG
    cmds = ["units real", "atom_style full", "boundary f f f",
            "special_bonds lj/coul 0.0 0.0 1.0",
            (f"pair_style hybrid/overlay {base} hbond/dreiding/lj 4 "
             f"{model.hbond_cutoff} {model.hbond_cutoff} {hb_ang}") if hb
            else f"pair_style {base}"]
    # a style is declared only when the molecule actually has that term
    if b_types:
        cmds.append("bond_style harmonic")
    styles = sorted({k[0] for k in a_types})
    if a_types:
        cmds.append(f"angle_style hybrid {' '.join(styles)}" if len(styles) > 1
                    else f"angle_style {styles[0]}")
    if d_types:
        cmds.append("dihedral_style harmonic")
    if i_types:
        cmds.append("improper_style umbrella")
    cmds.append("__DATA__")

    sub = f"{base.split()[0]} " if hb else ""   # the hybrid sub-style name
    for a in used:
        for b in used:
            if at_id[a] > at_id[b]:
                continue
            ia, ib = ti[a], ti[b]
            if ff.form == "x6":
                zz = [max(float(P["x6_zeta"][x]), 6.0 + 1e-6) for x in (ia, ib)]
                r0 = [float(P["vdw_r0"][x]) for x in (ia, ib)]
                dd = [float(P["vdw_d0"][x]) * EV_TO_KCAL for x in (ia, ib)]
                A = [dd[m] * 6.0 / (zz[m] - 6.0) * math.exp(zz[m]) for m in (0, 1)]
                C = [dd[m] * zz[m] / (zz[m] - 6.0) * r0[m] ** 6 for m in (0, 1)]
                c_ij = 0.5 * (zz[0] / r0[0] + zz[1] / r0[1])     # eq 35c
                cmds.append(f"pair_coeff {at_id[a]} {at_id[b]} {sub}"
                            f"{math.sqrt(A[0]*A[1]):.14g} {1.0/c_ij:.14g} "
                            f"{math.sqrt(C[0]*C[1]):.14g}")        # eq 35a,b
            else:
                d0 = math.sqrt(float(P["vdw_d0"][ia] * P["vdw_d0"][ib])) * EV_TO_KCAL
                r0a, r0b = float(P["vdw_r0"][ia]), float(P["vdw_r0"][ib])
                r0 = math.sqrt(r0a * r0b) if ff.combination == "geometric" \
                    else 0.5 * (r0a + r0b)                          # eq 36a/36c
                cmds.append(f"pair_coeff {at_id[a]} {at_id[b]} {sub}"
                            f"{d0:.14g} {r0 / 2.0 ** (1.0/6.0):.14g}")
    if hb:
        dt = {top.types[int(x)] for x in model.hbond_index[0]}
        ht = {top.types[int(x)] for x in model.hbond_index[1]}
        ac = {top.types[int(x)] for x in model.hbond_index[2]}
        for d in sorted(dt):
            for a in sorted(ac):
                for h in sorted(ht):
                    cmds.append(
                        f"pair_coeff {min(at_id[d], at_id[a])} "
                        f"{max(at_id[d], at_id[a])} hbond/dreiding/lj "
                        f"{at_id[h]} i {float(P['hbond_d0'])*EV_TO_KCAL:.14g} "
                        f"{float(P['hbond_r0']):.14g} 4 {model.hbond_cutoff} "
                        f"{model.hbond_cutoff} {hb_ang}")
    cmds += [f"bond_coeff {t} {k[0]} {k[1]}" for k, t in b_types.items()]
    multi = len(styles) > 1
    for k, t in a_types.items():
        args = f"{k[1]} {k[2]}" if k[0] == "cosine/squared" else f"{k[1]}"
        cmds.append(f"angle_coeff {t} {k[0] + ' ' if multi else ''}{args}")
    cmds += [f"dihedral_coeff {t} {k[0]} {k[1]} {k[2]}" for k, t in d_types.items()]
    cmds += [f"improper_coeff {t} {k[0]} {k[1]}" for k, t in i_types.items()]
    return "\n".join(L) + "\n", cmds


def run_lammps(model, positions):
    '''Per-term energies (kcal/mol) and forces from LAMMPS.'''
    from lammps import lammps
    data, cmds = lammps_twin(model, positions)
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "system.data")
        open(path, "w").write(data)
        lmp = lammps(cmdargs=["-log", "none", "-screen", "none", "-nocite"])
        for c in cmds:
            lmp.command(f"read_data {path}" if c == "__DATA__" else c)
        lmp.command("neigh_modify delay 0 every 1 check no one 10000 page 200000")
        lmp.command("thermo_style custom step ebond eangle edihed eimp evdwl ecoul pe")
        lmp.command("run 0 post no")
        out = {k: lmp.get_thermo(k) for k in
               ("ebond", "eangle", "edihed", "eimp", "evdwl", "ecoul", "pe")}
        # LAMMPS keeps atoms in its own internal order: map back through the ids
        f = np.array(lmp.numpy.extract_atom("f"), dtype=float).copy()
        ids = np.array(lmp.numpy.extract_atom("id"), dtype=int).copy()
        out["forces"] = f[np.argsort(ids)]
        lmp.close()
    return out


def run_xnn(model, positions, z):
    '''The same quantities from xnn, in the same units and keys.'''
    g = structure_to_graph(
        {"pos": torch.as_tensor(positions, dtype=torch.float64),
         "atomic_numbers": torch.as_tensor(z, dtype=torch.long)},
        cutoff=float(model.cutoff))
    o = ForceStressOutput(model)(g)
    e = {k: float(o[k]) * EV_TO_KCAL for k in
         ("e_bond", "e_angle", "e_torsion", "e_inversion", "e_vdw",
          "e_coulomb", "e_hbond")}
    return {"ebond": e["e_bond"], "eangle": e["e_angle"],
            "edihed": e["e_torsion"], "eimp": e["e_inversion"],
            # LAMMPS folds the hydrogen bond into the pair energy
            "evdwl": e["e_vdw"] + e["e_hbond"], "ecoul": e["e_coulomb"],
            "pe": float(o["energy"]) * EV_TO_KCAL,
            "forces": o["forces"].detach().numpy() * EV_TO_KCAL}

print("LAMMPS twin ready")

LAMMPS twin ready


## 3. Nine molecules covering every rule branch

Each structure is randomly displaced so that no term sits at its minimum:
bonds are stretched, angles bent and torsions twisted, which is what makes
the comparison informative. Between them the cases exercise every torsion
rule that generates a term, both angle forms, planar and non-planar
inversion centers, electrostatics and the explicit hydrogen bond.

In [4]:
def jit(p, s=0.10):
    return np.asarray(p, float) + s * rng.standard_normal((len(p), 3))


def ethane():
    r_cc, r_ch, ang = 1.53, 1.09, math.radians(109.471)
    s, c = r_ch * math.sin(math.pi - ang), r_ch * math.cos(math.pi - ang)
    pos = [[0, 0, 0], [r_cc, 0, 0]]
    for k in range(3):
        a = 2 * math.pi * k / 3
        pos.append([-c, s * math.cos(a), s * math.sin(a)])
    for k in range(3):
        a = 2 * math.pi * k / 3 + math.radians(47.0)
        pos.append([r_cc + c, s * math.cos(a), s * math.sin(a)])
    return jit(pos), [6, 6] + [1] * 6, ["C_3", "C_3"] + ["H_"] * 6, \
        [(0, 1), (0, 2), (0, 3), (0, 4), (1, 5), (1, 6), (1, 7)], [1.0] * 7


def ethylene():
    d, dh, ang = 1.34, 1.08, math.radians(120.0)
    x, y = dh * math.cos(math.pi - ang), dh * math.sin(math.pi - ang)
    pos = [[0, 0, 0], [d, 0, 0], [-x, y, 0], [-x, -y, 0], [d + x, y, 0], [d + x, -y, 0]]
    return jit(pos), [6, 6, 1, 1, 1, 1], ["C_2", "C_2"] + ["H_"] * 4, \
        [(0, 1), (0, 2), (0, 3), (1, 4), (1, 5)], [2.0, 1.0, 1.0, 1.0, 1.0]


def benzene():
    rc, rh = 1.39, 1.02
    pos = [[rc * math.cos(2 * math.pi * k / 6), rc * math.sin(2 * math.pi * k / 6), 0]
           for k in range(6)]
    pos += [[(rc + rh) * math.cos(2 * math.pi * k / 6),
             (rc + rh) * math.sin(2 * math.pi * k / 6), 0] for k in range(6)]
    bonds = [(k, (k + 1) % 6) for k in range(6)] + [(k, 6 + k) for k in range(6)]
    return jit(pos), [6] * 6 + [1] * 6, ["C_R"] * 6 + ["H_"] * 6, bonds, \
        [1.5] * 6 + [1.0] * 6


def biphenyl():
    rc, rh, link = 1.39, 1.02, 1.49
    pos, types, bonds, orders = [], [], [], []
    for xoff, kl, tw in ((0.0, 0, 0.0), (rc + link + rc, 3, 0.7)):
        base = len(pos)
        for k in range(6):
            a = 2 * math.pi * k / 6
            pos.append([xoff + rc * math.cos(a), rc * math.sin(a) * math.cos(tw),
                        rc * math.sin(a) * math.sin(tw)])
            types.append("C_R")
        bonds += [(base + k, base + (k + 1) % 6) for k in range(6)]
        orders += [1.5] * 6
        for k in range(6):
            if k == kl:
                continue                      # the linking carbon carries no H
            a, r = 2 * math.pi * k / 6, rc + rh
            pos.append([xoff + r * math.cos(a), r * math.sin(a) * math.cos(tw),
                        r * math.sin(a) * math.sin(tw)])
            types.append("H_")
            bonds.append((base + k, len(pos) - 1))
            orders.append(1.0)
    bonds.append((0, 14))                     # the exocyclic single bond, rule f
    orders.append(1.0)
    return jit(pos, 0.05), [6 if t == "C_R" else 1 for t in types], types, bonds, orders


def methanol():
    pos = [[0, 0, 0], [1.43, 0, 0], [1.80, 0.93, 0],
           [-0.4, 1.03, 0], [-0.4, -0.5, 0.9], [-0.4, -0.5, -0.9]]
    return jit(pos), [6, 8, 1, 1, 1, 1], \
        ["C_3", "O_3", "H__HB", "H_", "H_", "H_"], \
        [(0, 1), (1, 2), (0, 3), (0, 4), (0, 5)], [1.0] * 5


def water_dimer():
    pos = [[0, 0, 0], [0.96, 0, 0], [-0.24, 0.93, 0],
           [2.85, 0, 0], [3.15, 0.45, 0.80], [3.15, 0.45, -0.80]]
    return jit(pos, 0.05), [8, 1, 1, 8, 1, 1], \
        ["O_3", "H__HB", "H__HB", "O_3", "H__HB", "H__HB"], \
        [(0, 1), (0, 2), (3, 4), (3, 5)], [1.0] * 4


def carbon_dioxide():
    return jit([[0, 0, 0], [1.16, 0, 0], [-1.16, 0, 0]], 0.07), [6, 8, 8], \
        ["C_1", "O_1", "O_1"], [(0, 1), (0, 2)], [2.0, 2.0]


def hydrogen_peroxide():
    return jit([[0, 0, 0], [1.45, 0, 0], [-0.3, 0.9, 0.3], [1.75, 0.5, 0.85]], 0.05), \
        [8, 8, 1, 1], ["O_3", "O_3", "H__HB", "H__HB"], \
        [(0, 1), (0, 2), (1, 3)], [1.0] * 3


def propene():
    pos = [[0, 0, 0], [1.34, 0, 0], [2.15, 1.25, 0], [-0.55, -0.93, 0],
           [-0.55, 0.93, 0], [1.9, -0.93, 0], [1.6, 2.1, 0.4],
           [2.9, 1.2, 0.8], [2.6, 1.4, -0.98]]
    return jit(pos, 0.06), [6, 6, 6] + [1] * 6, ["C_2", "C_2", "C_3"] + ["H_"] * 6, \
        [(0, 1), (1, 2), (0, 3), (0, 4), (1, 5), (2, 6), (2, 7), (2, 8)], \
        [2.0] + [1.0] * 7


CASES = {
    "ethane (rule a)": (ethane, {}),
    "ethylene (rule c, inversions)": (ethylene, {}),
    "benzene (rule d, inversions)": (benzene, {}),
    "biphenyl (rule f)": (biphenyl, {}),
    "methanol + charges (eq 37)": (methanol,
        dict(charges=[0.145, -0.683, 0.418, 0.04, 0.04, 0.04])),
    "water dimer (eq 38 H-bond)": (water_dimer,
        dict(charges=[-0.66, 0.33, 0.33, -0.66, 0.33, 0.33])),
    "CO2 (linear angles, eq 10')": (carbon_dioxide, {}),
    "H2O2 (rule h)": (hydrogen_peroxide, {}),
    "propene (rules b + j)": (propene, {}),
}
print(f"{len(CASES)} molecules; torsion rules exercised:")
for name, (fn, kw) in CASES.items():
    pos, z, types, bonds, orders = fn()
    top = MolecularTopology.from_bonds(types, bonds, bond_orders=orders)
    m = Dreiding("dreiding", top, cutoff=14.0, **kw)
    rules = sorted({RULE_IDS[int(r)] for r in m.dihedral_rule})
    print(f"   {name:32s} {m.dihedral_index.shape[1]:3d} torsions {rules}, "
          f"{m.inversion_index.shape[1] // 3} inversion centers")

9 molecules; torsion rules exercised:
   ethane (rule a)                    9 torsions ['a'], 0 inversion centers
   ethylene (rule c, inversions)      4 torsions ['c'], 2 inversion centers
   benzene (rule d, inversions)      24 torsions ['d'], 6 inversion centers
   biphenyl (rule f)                 52 torsions ['d', 'f'], 12 inversion centers
   methanol + charges (eq 37)         3 torsions ['a'], 0 inversion centers
   water dimer (eq 38 H-bond)         0 torsions [], 0 inversion centers
   CO2 (linear angles, eq 10')        0 torsions [], 0 inversion centers
   H2O2 (rule h)                      1 torsions ['h'], 0 inversion centers
   propene (rules b + j)             10 torsions ['b', 'c', 'j'], 2 inversion centers


## 4. Term-by-term parity

Every molecule is evaluated in both nonbond forms. The comparison is per
term rather than on the total, so a compensating pair of errors cannot hide.

In [5]:
KEYS = ("ebond", "eangle", "edihed", "eimp", "evdwl", "ecoul", "pe")
rows, worst_e, worst_f = [], 0.0, 0.0
for name, (fn, kw) in CASES.items():
    pos, z, types, bonds, orders = fn()
    top = MolecularTopology.from_bonds(types, bonds, bond_orders=orders)
    for form, label in (("dreiding", "LJ"), ("dreiding/X6", "X6")):
        m = Dreiding(form, top, cutoff=14.0, **kw)
        a, b = run_xnn(m, pos, z), run_lammps(m, pos)
        de = max(abs(a[k] - b[k]) for k in KEYS)
        df = float(np.abs(a["forces"] - b["forces"]).max())
        worst_e, worst_f = max(worst_e, de), max(worst_f, df)
        rows.append((name, label, a["pe"], b["pe"], de, df))

print(f"{'molecule':32s} {'vdw':4s} {'E xnn':>13s} {'E LAMMPS':>13s} "
      f"{'max|dE| term':>13s} {'max|dF|':>10s}")
print("-" * 92)
for name, label, ea, eb, de, df in rows:
    print(f"{name:32s} {label:4s} {ea:13.6f} {eb:13.6f} {de:13.2e} {df:10.2e}")
print("-" * 92)
print(f"worst over {len(rows)} comparisons: energy {worst_e:.2e} kcal/mol, "
      f"forces {worst_f:.2e} kcal/mol/A")
assert worst_e < 1e-7 and worst_f < 1e-6
print("\nPASS: every term agrees to within double-precision round-off")

molecule                         vdw          E xnn      E LAMMPS  max|dE| term    max|dF|
--------------------------------------------------------------------------------------------
ethane (rule a)                  LJ       70.226193     70.226193      7.11e-11   4.07e-10
ethane (rule a)                  X6       68.306646     68.306646      6.82e-11   4.05e-10
ethylene (rule c, inversions)    LJ      108.015624    108.015624      8.66e-11   3.69e-10
ethylene (rule c, inversions)    X6      107.619838    107.619838      8.71e-11   3.69e-10
benzene (rule d, inversions)     LJ      130.889244    130.889244      4.12e-11   4.03e-10
benzene (rule d, inversions)     X6      122.004611    122.004611      4.12e-11   4.03e-10
biphenyl (rule f)                LJ      116.353479    116.353479      1.47e-10   3.87e-10
biphenyl (rule f)                X6      104.223858    104.223858      1.60e-10   3.86e-10
methanol + charges (eq 37)       LJ       56.517693     56.517693      5.67e-11   4.22e-

The residuals sit at the level of double-precision accumulation over the
term lists (the coefficients are written to the data file with 14
significant digits), which is what parity between two independent
implementations of the same closed-form expressions should look like.

A per-term breakdown of one case makes the comparison concrete.

In [6]:
pos, z, types, bonds, orders = benzene()
top = MolecularTopology.from_bonds(types, bonds, bond_orders=orders)
m = Dreiding("dreiding", top, cutoff=14.0)
a, b = run_xnn(m, pos, z), run_lammps(m, pos)
print("benzene, randomized geometry (kcal/mol)\n")
print(f"{'term':28s} {'xnn':>14s} {'LAMMPS':>14s} {'difference':>12s}")
for key, label in (("ebond", "bonds (eq 4a)"), ("eangle", "angles (eq 10a)"),
                   ("edihed", "torsions (eq 17, rule d)"),
                   ("eimp", "inversions (eq 28c)"),
                   ("evdwl", "van der Waals (eq 31')"),
                   ("ecoul", "electrostatics (eq 37)"), ("pe", "total")):
    print(f"{label:28s} {a[key]:14.8f} {b[key]:14.8f} {a[key]-b[key]:12.2e}")
print(f"\nmax force difference: {np.abs(a['forces']-b['forces']).max():.2e} kcal/mol/A")

benzene, randomized geometry (kcal/mol)

term                                    xnn         LAMMPS   difference
bonds (eq 4a)                   71.39133375    71.39133375     1.02e-10
angles (eq 10a)                 33.39396596    33.39396596    -1.44e-11
torsions (eq 17, rule d)         4.90979626     4.90979626     4.12e-11
inversions (eq 28c)              5.74512453     5.74512453    -6.73e-12
van der Waals (eq 31')          11.24619223    11.24619223    -1.29e-11
electrostatics (eq 37)           0.00000000     0.00000000     0.00e+00
total                          126.68641273   126.68641273     1.10e-10

max force difference: 4.00e-10 kcal/mol/A


## 5. Closed-form checks against the paper

Two quantities in DREIDING have exact analytic values that any correct
implementation must hit, independent of any other code.

**The ethane torsion barrier.** Eq 14 gives $V_{JK} = 2.0$ kcal/mol as the
*total* barrier of an sp3-sp3 single bond, split over the nine $I,L$
combinations ("the program uses a barrier of $V_{IJKL} = 2/9$ for each of
the nine possibilities"). At an ideal tetrahedral geometry every bond and
angle sits at its minimum, so the torsion term is the whole story and the
eclipsed-staggered difference must be exactly 2.0.

**The hydrogen bond.** Eq 38 has its minimum at $R = R_{hb}$ with depth
$-D_{hb}$ when $\theta_{DHA} = 180°$, since $5 - 6 = -1$ and $\cos^4\theta
= 1$ there.

In [7]:
def ideal_ethane(phi_deg):
    r_cc, r_ch, ang = 1.53, 1.09, math.radians(109.471)
    s, c = r_ch * math.sin(math.pi - ang), r_ch * math.cos(math.pi - ang)
    pos = [[0, 0, 0], [r_cc, 0, 0]]
    for k in range(3):
        a = 2 * math.pi * k / 3
        pos.append([-c, s * math.cos(a), s * math.sin(a)])
    for k in range(3):
        a = 2 * math.pi * k / 3 + math.radians(phi_deg)
        pos.append([r_cc + c, s * math.cos(a), s * math.sin(a)])
    return pos


top = MolecularTopology.from_bonds(
    ["C_3", "C_3"] + ["H_"] * 6,
    [(0, 1), (0, 2), (0, 3), (0, 4), (1, 5), (1, 6), (1, 7)],
    bond_orders=[1.0] * 7)
m = Dreiding("dreiding", top, cutoff=14.0)
z = [6, 6] + [1] * 6
print(f"ethane: {m.dihedral_index.shape[1]} torsion terms, each weighted "
      f"{float(m.dihedral_weight[0]):.6f} = 1/9 of V_JK\n")
print(f"{'phi (deg)':>10s} {'E_torsion':>12s} {'1/2 V (1+cos 3phi)':>20s}")
for phi in (0.0, 17.0, 41.0, 60.0, 88.0, 120.0):
    g = structure_to_graph({"pos": torch.tensor(ideal_ethane(phi)),
                            "atomic_numbers": torch.tensor(z)}, cutoff=14.0)
    out = m(g)
    want = 0.5 * 2.0 * (1.0 + math.cos(3.0 * math.radians(phi)))
    print(f"{phi:10.1f} {float(out['e_torsion'])*EV_TO_KCAL:12.9f} {want:20.9f}")
g0 = structure_to_graph({"pos": torch.tensor(ideal_ethane(0.0)),
                         "atomic_numbers": torch.tensor(z)}, cutoff=14.0)
g6 = structure_to_graph({"pos": torch.tensor(ideal_ethane(60.0)),
                         "atomic_numbers": torch.tensor(z)}, cutoff=14.0)
barrier = (float(m(g0)["e_torsion"]) - float(m(g6)["e_torsion"])) * EV_TO_KCAL
print(f"\neclipsed - staggered torsion energy = {barrier:.12f} kcal/mol "
      f"(eq 14: exactly 2.0)")
assert abs(barrier - 2.0) < 1e-10

ethane: 9 torsion terms, each weighted 0.111111 = 1/9 of V_JK

 phi (deg)    E_torsion   1/2 V (1+cos 3phi)
       0.0  2.000000000          2.000000000
      17.0  1.629320391          1.629320391
      41.0  0.455360965          0.455360965
      60.0  0.000000000          0.000000000
      88.0  0.895471537          0.895471537
     120.0  2.000000000          2.000000000

eclipsed - staggered torsion energy = 2.000000000000 kcal/mol (eq 14: exactly 2.0)


In [8]:
top = MolecularTopology.from_bonds(
    ["O_3", "H__HB", "H__HB", "O_3", "H__HB", "H__HB"],
    [(0, 1), (0, 2), (3, 4), (3, 5)], bond_orders=[1.0] * 4)
m = Dreiding("dreiding", top, cutoff=16.0)
lib = read_dreiding("dreiding")
print(f"{'R(O-O)':>8s} {'E_hbond':>12s}   (D_hb = {lib.hbond_d0}, R_hb = {lib.hbond_r0})")
for r_oo in (2.40, 2.75, 3.00, 3.50, 4.50):
    # a strictly linear O-H...O bridge: the donor H lies on the O-O axis
    pos = [[0, 0, 0], [0.96, 0, 0], [-0.24, 0.93, 0],
           [r_oo, 0, 0], [r_oo + 0.3, 0.90, 0], [r_oo + 0.3, -0.90, 0]]
    g = structure_to_graph({"pos": torch.tensor(pos, dtype=torch.float64),
                            "atomic_numbers": torch.tensor([8, 1, 1, 8, 1, 1])},
                           cutoff=16.0)
    e = float(m(g)["e_hbond"]) * EV_TO_KCAL
    print(f"{r_oo:8.2f} {e:12.8f}" + ("   <- eq 38 minimum, -D_hb" if abs(r_oo - 2.75) < 1e-9 else ""))

  R(O-O)      E_hbond   (D_hb = 9.0, R_hb = 2.75)
    2.40  19.82676470
    2.75  -9.00000000   <- eq 38 minimum, -D_hb
    3.00  -6.78100669
    3.50  -2.35108885
    4.50  -0.27019660


## 6. Summary

| check | result |
|---|---|
| Tables I, II, III, V of the paper | every generator reproduced exactly |
| eqs 14-23, the nine torsion rules | present with the published $(V, n, \phi_0)$ |
| LAMMPS parity, 9 molecules x 2 nonbond forms, term by term | energies $<10^{-7}$ kcal/mol, forces $<10^{-6}$ kcal/mol/Å |
| eq 14, the ethane torsion barrier | exactly 2.0 kcal/mol |
| eq 38, the hydrogen-bond minimum | exactly $-D_{hb}$ at $R = R_{hb}$ |

The rule engine and every energy expression therefore agree with an
independent implementation to round-off, and the parameters agree with the
published tables. The physical consequences -- relaxed rotational barriers
(Table XI) and conformational energies (Table XII) -- are reproduced in
`examples/ffnn/dreiding/dreiding_conformational_energetics.ipynb`.